In [ ]:
# - Choose the model whose feature selection you want to run locally.
# - Paste an execution ID from your configured training pipeline, or its full ARN.
# - These inputs do nothing until you run the preparation cell.
MODEL_ID = "usecase2"
EXECUTION_ID = ""


# Run feature selection locally

Run the setup cell, then the short execution cell. The helper reads the actual
FeatureSelection job and its raw data, including cached producer paths. Selection
uses **your current checkout's code and parameters**, in the notebook kernel.
The setup itself does not run selection.

The execution cell imports `mlops_models.entrypoints.feature_selection` and calls
`feature_selection.execute(...)` directly. The support helper handles input and
environment setup only.

The execution cell prints a complete **VS Code main block**, even if the model
raises an exception or you interrupt it. Copy that output to the bottom of
`mlops-models/src/mlops_models/entrypoints/feature_selection.py`, replacing the
previous local main block. Run that file with this checkout installed editable
in your selected Python interpreter. The existing module-based F5 configuration
also works; `launch.json` is optional. See the
[root README](../../README.md#run-a-step-locally) for setup.
The notebook itself supplies its source path and does not use `launch.json`.

In the generated VS Code block, edit `MODEL_ID` and `EXECUTION_ID` at the top
to switch models/runs without returning to this notebook. The block reads the
selected job and refreshes its inputs and metadata each time. A bare execution ID
uses this pipeline; use a full ARN to select another pipeline or Region.

Rerun the execution cell after editing base/model selection code. Rerun setup
after changing the model/execution inputs. Restart the kernel for shared
dependency or checkout changes. Your notebook needs the model's dependencies.
The setup and export code live in [the helper](../source/feature_selection/support.py).


In [ ]:
# - Resolve the selected job and download/reuse its raw input.
# - Prepare a fresh output folder and a standalone VS Code export.
# - Keep selection unexecuted until the next cell.
step = None
from pathlib import Path
import runpy

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
support = runpy.run_path(str(ROOT / "notebooks/source/feature_selection/support.py"))
step = support["prepare"](ROOT, MODEL_ID, EXECUTION_ID)


In [ ]:
# - Import and call the real feature-selection entrypoint directly.
# - Keep the original traceback if selection fails; restore the environment.
# - Always print the prepared VS Code block, including after interruption.
import importlib

result = None
if step is None:
    raise ValueError("Run the preparation cell successfully first.")
try:
    with support["local_environment"](step):
        from mlops_models.entrypoints import feature_selection

        feature_selection = importlib.reload(feature_selection)
        print("Entrypoint:", feature_selection.__file__, flush=True)
        result = feature_selection.execute(
            model_id=step.model_id,
            input_path=step.data_directory,
            output_dir=step.output_directory,
        )
    print("Returned value:", result)
finally:
    print(step.vscode_code, flush=True)
